# Convective Environment Statistics by Lightning/OT Category (Land vs Ocean)

Purpose: Compute distributions of convective environment variables (MUCAPE, low-level
moisture, lapse rates, etc.) for each lightning/OT co-occurrence category — "OT only",
"OT + LS", "LS only" — separately for land and ocean, to characterize the environmental
conditions associated with each category across warm seasons (2016–2024).

Input:
- Categorized lightning/OT point CSVs (from 04_categorical_lightning_OT_dilation.ipynb):
  ls_only_land_3d.csv, ls_only_ocean_3d.csv, ls_ot_land_3d.csv, ls_ot_ocean_3d.csv,
  ot_only_land_3d.csv, ot_only_ocean_3d.csv
- BARRA-R2 hourly reanalysis variables: MUCAPE, MULPLmixr, MULCL, LR36, RH36mean, MULPLtemp
- Region of interest: Northern Australia (lat: -20 to -10, lon: 120 to 148, +2° pad for
  file loading)
- Warm season: October-April, years 2016-2024

Processing:
1. Load categorized lightning/OT CSVs and group into three categories (OT only, OT + LS,
   LS only), each split by land/ocean.
2. For each environment variable, load matching monthly NetCDF files for warm-season
   months across all years, spatially subsetting with a buffer.
3. Mask each variable's data to convective conditions using MUCAPE > 0.
4. For each category/contrast combination, extract the variable's value at the nearest
   grid point/time matching each categorized lightning/OT detection.
5. Compile extracted values into a long-format dataframe (group, contrast, value,
   variable, year) and save per-variable to feather.
6. Compute summary statistics (quartiles, median, mean, IQR, whiskers) per group/contrast
   for boxplot-style comparisons.

Output:
- Per-variable extracted value dataframes:
  /g/data/if69/tp4064/Project_1/categorical_position/Box_plot/selected_dataset_<var>.feather
- Combined summary statistics table:
  /g/data/if69/tp4064/Project_1/categorical_position/Box_plot/stats_parameter.csv

In [1]:
# Standard libraries
import os
import glob

# Data manipulation
import pandas as pd
import numpy as np
import xarray as xr

# Regridding and spatial analysis
import xesmf as xe  # Regridding tool for geospatial data
import pyarrow

# Parallel computing
import dask

# import kaleido


##  Initialize Dask Client

We use Dask for parallel computing to speed up data processing.


In [2]:
from dask.distributed import Client

# Initialize Dask client with one thread per worker
client = Client(threads_per_worker=1)
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 28
Total threads: 28,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38659,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33201,Total threads: 1
Dashboard: /proxy/39153/status,Memory: 8.97 GiB
Nanny: tcp://127.0.0.1:33683,


In [3]:
def yearly_data_processing(years, DATA_DIR, var, CATEGORIES, warm_months=[10, 11, 12, 1, 2, 3, 4]):

    """
    extract variable data for given categories for each year and warm season only
    and at the end concat them together for statistical analysis
    """

    all_data=[]
    
    for year in years:
        
        start = f"{year}-01-01"
        end   = f"{year}-12-31"
        
        print(start, end)

    
        df_all = xr.open_mfdataset(
                get_files(
                    os.path.join(DATA_DIR, var, 'latest/'),
                    start,
                    end,
                    warm_months=[10, 11, 12, 1, 2, 3, 4]
                ),
                chunks={},  # or {'time': 240} later
                combine='by_coords',
                preprocess=preprocess_fn
            )
        data_mucape_mask = (xr.open_mfdataset(
                get_files(
                    os.path.join(DATA_DIR, 'MUCAPE', 'latest/'),
                    start,
                    end,
                    warm_months=[10, 11, 12, 1, 2, 3, 4]
                ),
                chunks={},  # or {'time': 240} later
                combine='by_coords',
                preprocess=preprocess_fn
            )['MUCAPE']>0)

        df_masked = df_all.where(data_mucape_mask)

        rows = []
        for group, (land_df, ocean_df) in CATEGORIES.items():
            for contrast, cat_df in [('Land', land_df), ('Ocean', ocean_df)]:
                vals = extract_values(df_masked, cat_df,year,warm_months, var)
                # CHECK = select_point_based_data(df_masked, cat_df, warm_months)
                # compute_stats(vals)
                rows += [{'group': group, 'contrast': contrast, 'value': v,'Variable':var, 'year': year} for v in vals]
    
        dataset = pd.DataFrame(rows)  

        all_data.append(dataset)

    df = pd.concat(all_data)

    return df



# ── Helper: match monthly NetCDF files in a date range ────────────────────────
def get_files(path: str, start: str, end: str, warm_months) -> list[str]:
    """
    building monthly filenames for warm season which need to be extracted
    """
    months = pd.date_range(start=start, end=end, freq='MS')#.strftime('%Y%m')
    months = months[months.month.isin(warm_months)].strftime('%Y%m')
    files  = sorted(f for m in months for f in glob.glob(os.path.join(path, f'*-{m}.nc')))
    return files

# ──
## Filter Data based on input given, specifically design for Land/Ocean Region for OT.
#This function filters the dataset to a specific lat/lon box and removes data from 2015.

def select_point_based_data(data, variable,year, warm_months=(10, 11, 12, 1, 2, 3, 4)):
    """
    Select nearest grid-point data for OT/lightning points.
    """

    # Copy to avoid modifying original dataframe
    variable = variable.copy()

    # Convert time once
    variable['time'] = pd.to_datetime(variable['time'])

    # Single combined mask (faster + cleaner)
    mask = (
        (variable['lat'].between(LAT_MIN, LAT_MAX)) &
        (variable['lon'].between(LON_MIN, LON_MAX)) &
        (variable['time'].dt.year == year) &
        (variable['time'].dt.month.isin(warm_months))
    )

    variable = variable.loc[mask]

    # Reduce precision slightly for cleaner nearest-neighbour matching
    variable['lat'] = variable['lat'].round(4)
    variable['lon'] = variable['lon'].round(4)

    # Convert directly to NumPy arrays (lighter than pandas Series)
    da_lat = xr.DataArray(variable['lat'].to_numpy(), dims='points')
    da_lon = xr.DataArray(variable['lon'].to_numpy(), dims='points')
    da_time = xr.DataArray(variable['time'].to_numpy(), dims='points')

    # Nearest selection
    data_select = data.sel(
        lat=da_lat,
        lon=da_lon,
        time=da_time,
        method='nearest'
    )

    return data_select


def preprocess_fn(ds):
    """
    preprocess the netcdf file, spatial selection and droping "crs" for efficiency
    """
    PAD = 2
    ds = (
        ds.drop_vars("crs", errors="ignore")
          .sel(
              lat=slice(LAT_MIN - PAD, LAT_MAX + PAD),
              lon=slice(LON_MIN - PAD, LON_MAX + PAD),
              # time=slice(, END_DATE),    # 
          )
    )

    
    return ds


def compute_stats(dataset):
    """
    takes dataframe (with group, contrast, value columns) and computes summary statistics per group/contrast: quartiles, median, mean, IQR, and boxplot-style whisker bounds.
    """
    grouped = dataset.groupby(['group', 'contrast'])
    summary = grouped['value'].agg(
        Q1      = lambda x: np.percentile(x, 25),
        Median  = 'median',
        Mean    = 'mean',
        Q3      = lambda x: np.percentile(x, 75),
        Min     = 'min',
        Max     = 'max',
    ).reset_index()
    summary['IQR']          = summary['Q3'] - summary['Q1']
    summary['Lower Whisker']= summary.apply(lambda r: max(r.Min, r.Q1 - 1.5 * r.IQR), axis=1)
    summary['Upper Whisker']= summary.apply(lambda r: min(r.Max, r.Q3 + 1.5 * r.IQR), axis=1)
    return summary

def extract_values(df_all, cat_df,year,warm_months, var):
    """
    extracting one variable at a time
    """
    vals = select_point_based_data(df_all, cat_df,year, warm_months)[var].values
    return list(vals[~np.isnan(vals)])


In [4]:
#input data
list_dir_1 = ['MUCAPE', 'MULPLmixr', 'MULCL', 'LR36', 'RH36mean', 'MULPLtemp']
LAT_MIN, LAT_MAX = -20, -10
LON_MIN, LON_MAX = 120, 148
DATA_DIR   = '/g/data/ob53/BARRA2/output/reanalysis/AUST-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/'
PAD = 2
warm_months = [10, 11, 12, 1, 2, 3, 4]
years = range(2016, 2025)
CSV_DIR = "/g/data/if69/tp4064/Project_1/categorical_position/Dialation/"

csv_files = {
    "ls_only_land"   : "ls_only_land_3d.csv",
    "ls_only_ocean"  : "ls_only_ocean_3d.csv",
    "ls_ot_land"     : "ls_ot_land_3d.csv",
    "ls_ot_ocean"    : "ls_ot_ocean_3d.csv",
    "land_ot_no_ls"  : "ot_only_land_3d.csv",
    "ocean_ot_no_ls" : "ot_only_ocean_3d.csv",
}


In [5]:
# processing the data
data = {name: pd.read_csv(CSV_DIR + file) for name, file in csv_files.items()}

CATEGORIES = {
    'OT only': (data['land_ot_no_ls'],  data['ocean_ot_no_ls']),
    'OT + LS': (data['ls_ot_land'],     data['ls_ot_ocean']),
    'LS only': (data['ls_only_land'],   data['ls_only_ocean']),}


all_stats = []

for var in list_dir_1 :

    print(var)

    df = yearly_data_processing(years, DATA_DIR, var, CATEGORIES, warm_months=[10, 11, 12, 1, 2, 3, 4])

    df.to_feather(f"/g/data/if69/tp4064/Project_1/categorical_position/Box_plot/1_selected_dataset_{var}.feather")

    stats = compute_stats(df)
    stats['variable'] = var
    
    all_stats.append(stats)


MUCAPE
2016-01-01 2016-12-31
2017-01-01 2017-12-31
2018-01-01 2018-12-31
2019-01-01 2019-12-31
2020-01-01 2020-12-31
2021-01-01 2021-12-31
2022-01-01 2022-12-31
2023-01-01 2023-12-31
2024-01-01 2024-12-31
MULPLmixr
2016-01-01 2016-12-31
2017-01-01 2017-12-31
2018-01-01 2018-12-31
2019-01-01 2019-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 11.58 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2020-01-01 2020-12-31
2021-01-01 2021-12-31
2022-01-01 2022-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 11.65 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2023-01-01 2023-12-31
2024-01-01 2024-12-31
MULCL
2016-01-01 2016-12-31
2017-01-01 2017-12-31


...

2022-01-01 2022-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 10.41 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2023-01-01 2023-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 11.10 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2024-01-01 2024-12-31


In [6]:
#saving th stats
(pd.concat(all_stats)).to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Box_plot/warm_season_stats_parameter.csv")